In [1]:
import matplotlib.pyplot as plt
import numpy as np

from examples.seismic import Model, plot_velocity, TimeAxis, RickerSource, Receiver
from devito import TimeFunction, VectorTimeFunction, Eq, solve, Operator
from devito.finite_differences.operators import div, grad
from matplotlib.animation import FuncAnimation
from devito import NODE

from lista_utils import *

# QUESTÃO 2

In [ ]:
# Construindo modelo de velocidade

nx, nz = 150, 150 # Quantidade de pontos nas direções X e Z (150 pontos)
dx, dz = 10, 10 # Espaçamento entre pontos nas direções X e Z (10 metros)
origin = (0., 0.)  # Coordenadas da origem do modelo
dtype = 'float32'
nbl = 10
space_order = 8

vp = np.ones((nx,nz), dtype=dtype) * 2 # Velocidade da primeira camada (2 km/s)
vp[:, nz//2 : ] = 3 # Velocidade da segunda camada (3 km/s)

rho = np.ones((nx,nz), dtype=dtype) * 2 # Densidade do modelo (2 g/cc)
rho[:, nz//2 : ] = 3 # Densidade da segunda camada (3 g/cc)
b = 1 / rho

model = Model(vp=vp, b=b, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs="damp")

# Plotando o modelo de velocidade

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet'}

fig, ax = plt.subplots(figsize=(6,4))

img = ax.imshow(model.vp.data.T, **plot_options)
ax.set_title('Modelo Vp')
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Profundidade (m)')

cbar = fig.colorbar(img)
cbar.set_label('Vp (km/s)')

fig.tight_layout()
plt.show()

In [ ]:
# Definindo a fonte sísmica (Ricker)

t0 = 0.  # Tempo inicial da modelagem t=0ms
tn = 2000.  # Tempo final da modelagem t=1000ms
dt = model.critical_dt  # Tempo entre iterações (2ms)
f0 = 0.01  # Frequência de pico da wavelet (20Hz = 0.020 kHz)
ns = 1

time_range = TimeAxis(start=t0, stop=tn, step=dt)
src = RickerSource(name='src', grid=model.grid, f0=f0, npoint=ns, time_range=time_range)

# Definindo coordenadas da fonte
src.coordinates.data[0, 0] = model.domain_size[0] * .5 # Centralizando no eixo x
src.coordinates.data[0, 1] = model.domain_size[0] * .5 # Centralizando no eixo y

# Plotando a fonte e coordenadas
print(f'Coordenadas: {src.coordinates.data}')

fig, ax = plt.subplots(figsize=(7,2))

ax.plot(src.time_values, src.data)
ax.set_xlabel('Tempo (ms)')
ax.set_ylabel('Amplitude')

fig.tight_layout()
plt.show()

In [ ]:
# Definindo os receptores

ng = nx

rec = Receiver(name='rec', grid=model.grid, npoint=ng, time_range=time_range)

# Definindo coordenadas dos receptores
rec.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], ng) # Definindo coordenadas equiespaçadas em x
rec.coordinates.data[:, 1] = 0.  # Profundidade dos receptores (0m)

# Plotting velocity model com fonte e receptores

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet'}

fig, ax = plt.subplots(figsize=(6,4))

img = ax.imshow(model.vp.data.T, **plot_options)
ax.scatter(rec.coordinates.data[::8,0], rec.coordinates.data[::8,1], c='green', marker='v')
ax.scatter(src.coordinates.data[0,0], src.coordinates.data[0,1], c='red', marker='x')
ax.set_title('Modelo Vp')
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Profundidade (m)')

cbar = fig.colorbar(img)
cbar.set_label('Vp (km/s)')

fig.tight_layout()
plt.show()

Partindo da equação da onda acústica de segunda ordem:

$$ \frac{\partial^2 P}{\partial t^2} - \kappa \nabla \cdot \frac{1}{\rho} \nabla P = 0$$

In [ ]:
P, rec = acoustic_modelling(model, src, rec)

# Plotando registro dos geofones

amax = max(rec_sigma.data.max(), abs(rec_sigma.data.min())) * .5
plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto', 'vmin':-amax, 'vmax':amax}

fig, ax = plt.subplots(figsize=(5,7))

ax.imshow(rec.data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

## a)

In [ ]:
# Plotando snapshots

plot_t0 = 100
plot_tn = 2000
plot_dt = 20

plot_snaps(P, model, src, plot_t0, plot_tn, plot_dt, cols=6)

## b)

In [ ]:
# Plotando filmagem da propagação

# OBS01: ESTA CÉLULA DEMORA 2min PARA EXECUTAR
# OBS02: PARA EXECUTAR O VÍDEO, BASTA DAR PLAY NO WIDGET QUE APARECERÁ

output = plot_video(P, rec, model, interval=1)

output

## c)

A amplitude da onda diminui com o passar do tempo devido a três principais fatores: divergência esférica, reflexões e amortecimento da borda. O primeiro fator é um fator físico e acontece na realidade. A perda de energia por divergência esférica se dá devido à propagação radial/esférica da onda. A energia inicial da onda, concentrada em aproximadamente um ponto, é a única responsável por sua propagação (energia total) e à medida em que a onda se propaga essa mesma energia, antes concentrada em um ponto, deve se dividir em uma frente de onda esférica cujo raio aumenta com o tempo.

De acordo com a equação da área da superfície de uma esfera

$$ A=4 \pi R^2 $$

podemos dizer que a energia da onda diminui com o quadrado do raio.

Diferentemente do primeiro experimento, o modelo estratificado é responsável por causar reflexões, dividindo a energia da onda entre as frentes de onda transmitida, refletida e refratada.

Outro fator impactante para a diminuição da amplitude na modelagem é o amortecimento da borda. Em casos de simulação computacional, deve ser adicionada ao modelo uma borda atenuante para que a simulação se torne mais coerente com a realidade (modelo infinito), afinal na Terra não há bordas. Portanto, grande parte da atenuação da onda nessa modelagem se deu devido à borda atenuante.

## d)

\begin{cases}
    \rho \frac{\partial \mathbf{V}}{\partial t} - \nabla P = 0 \\ \\
    \frac{\partial P}{\partial t} - \kappa \nabla \cdot \mathbf{V} = \mathbf{f}
\end{cases}

In [ ]:
# Definindo os receptores

ng = nx

rec_vx = Receiver(name='rec_vx', grid=model.grid, npoint=ng, time_range=time_range)
rec_vz = Receiver(name='rec_vz', grid=model.grid, npoint=ng, time_range=time_range)
rec_p = Receiver(name='rec_p', grid=model.grid, npoint=ng, time_range=time_range)

# Prescribe even spacing for receivers along the x-axis
rec_vx.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=ng)
rec_vx.coordinates.data[:, 1] = 20.  # postion of receiver at 400 m depth for vx

rec_vz.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=ng)
rec_vz.coordinates.data[:, 1] = 20.  # postion of receiver at 400 m depth for vz

rec_p.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=ng)
rec_p.coordinates.data[:, 1] = 20. # postion of receiver at 10 m of depth for pressure field

plot_aquisition_setup(model, src, rec_p)

In [ ]:
# Plotando sismogramas

rec = [rec_vx, rec_vz, rec_p]

V, P, rec = acoustic_modelling(model, src, rec, order=1)

plot_options = {'aspect':'auto', 'cmap':'Greys'}

fig, axes = plt.subplots(1, 3, figsize=(12,7))

axes[0].imshow(rec[0].data, **plot_options)
axes[0].set_title('Receptores (Vx)')
axes[1].imshow(rec[1].data, **plot_options)
axes[1].set_title('Receptores (Vz)')
axes[2].imshow(rec[2].data, **plot_options)
axes[2].set_title('Receptores (Pressão)')

fig.tight_layout()
plt.show()

In [ ]:
# Plotando snapshots

plot_snaps(P, model, src, 100, 2000, 20, cols=6)

# e)

In [ ]:
# Configurando aquisição

# Definindo coordenadas da fonte
src.coordinates.data[0, 1] = 0. # Profundidade da fonte (0m)

ng = nx

rec_inv = Receiver(name='rec', grid=model.grid, npoint=ng, time_range=time_range)

# Definindo coordenadas dos receptores
rec_inv.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], ng) # Definindo coordenadas equiespaçadas em x
rec_inv.coordinates.data[:, 1] = 0.  # Profundidade dos receptores (0m)

print(f'Coordenadas da fonte: \n\tx: {src.coordinates.data[0,0]}m \n\tz: {src.coordinates.data[0,1]}m')
print(f'N° de Receptores: {len(rec_inv.coordinates.data)}')

In [ ]:
P, rec = acoustic_modelling(model, src, rec, order=2)

# Plotando registro dos geofones

plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto', 'vmax':rec.data.max() * 0.3}

fig, ax = plt.subplots(figsize=(5,7))

ax.imshow(rec.data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

In [ ]:
# Plotando snapshots

plot_snaps(P, model, src, 100, 2000, 20, cols=6)

## f)

In [ ]:
# Construindo modelo de velocidade

vp = np.ones((nx,nz), dtype=dtype) * 3 # Velocidade da primeira camada (3 km/s)
vp[:, nz//2 : ] = 2 # Velocidade da segunda camada (2 km/s)

rho = np.ones((nx,nz), dtype=dtype) * 3 # Densidade do modelo (3 g/cc)
rho[:, nz//2 : ] = 2 # Densidade da segunda camada (2 g/cc)
b = 1 / rho

model_inv = Model(vp=vp, b=b, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs="damp")

# Plotando o modelo de velocidade

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet'}

fig, ax = plt.subplots(figsize=(6,4))

img = ax.imshow(model_inv.vp.data.T, **plot_options)
ax.set_title('Modelo Vp')
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Profundidade (m)')

cbar = fig.colorbar(img)
cbar.set_label('Vp (km/s)')

fig.tight_layout()
plt.show()

In [ ]:
P, rec = acoustic_modelling(model_invert, src, rec, order=2)

# Plotando registro dos geofones

plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto', 'vmax':rec.data.max() * 0.3}

fig, ax = plt.subplots(figsize=(5,7))

ax.imshow(rec.data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

In [ ]:
# Plotando snapshots

plot_snaps(P, model_inv, src, 100, 2000, 20, cols=6)

## g)

In [ ]:
P, rec = acoustic_modelling(model, src, rec, order=2)
P_inv, rec_inv = acoustic_modelling(model_inv, src, rec_inv, order=2)

plot_dt = 200
plot_t0 = 200
plot_tn = 600

plot_snaps(P, model, src, plot_t0, plot_tn, plot_dt, cols=4)
plot_snaps(P_inv, model_inv, src, plot_t0, plot_tn, plot_dt, cols=4)

# Plotando Sismograma
plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'vmax':rec.data.max() * 0.3}

fig, axes = plt.subplots(1, 2, figsize=(12,17))

axes[0].imshow(rec_p.data, **plot_options)
axes[0].set_xlabel('Distância (m)')
axes[0].set_ylabel('Tempo (ms)')

axes[1].imshow(rec_inv.data, **plot_options)
axes[1].set_xlabel('Distância (m)')
axes[1].set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

De acordo com a Lei de Snell, quando a onda muda de um determinado meio para outro de maior velocidade, como no primeiro experimento, a trajetória do raio de onda também muda e o ângulo de transmissão aumenta com relação ao de incidência, formando uma frente de onda com menor concavidade. Nesse cenário, passa a existir um ângulo crítico de incidência no qual a frente de onda não transmite mais e refrata (caminha sobre a superfície de contato entre os meios). Já no último caso, com as velocidades invertidas, a onda muda de um meio para outro meio de menor velocidade. Com isso, os raios de onda transmitidos se aproximam da reta normal à superfície entre as camadas, impossibilitando a refração e formando uma frente de onda com maior concavidade.

In [ ]:
# --------- FIM --------- #